# Sandbox de Exploração - Modelo M1 (SARIMA)
Objetivo: Testar a previsão de tendências (Séries Temporais) utilizando SARIMA/ARIMA, incorporando conceitos do Projeto_acabado.R

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Credenciais e Host
_PG_HOST = os.environ.get("PG_HOST", "localhost")
_PG_PORT = os.environ.get("PG_PORT", "5432")

# 1. Adicionado '+psycopg2' para consistência
URL_SANDBOX = f"postgresql+psycopg2://ae_analista:senha_super_segura@{_PG_HOST}:{_PG_PORT}/auto_escala"

# 2. Definir o search_path para apontar automaticamente para a Sandbox e ler do DW
engine = create_engine(
    URL_SANDBOX, 
    connect_args={'options': '-c search_path=auto_escala_sandbox,auto_escala_dw'}
)

print("Engine conectada à Sandbox com sucesso!")


Engine connectada à Sandbox.


In [2]:
# Vamos carregar dados de fact_trends diretamente para testar o modelo M1
query = """
    SELECT
        ft.marca_key, ft.tipo_key, ft.combustivel_key, ft.localizacao_key,
        dtp.ano, dtp.mes,
        AVG(ft.valor_interesse)  AS valor
    FROM auto_escala_dw.fact_trends ft
    JOIN auto_escala_dw.dim_tempo dtp ON ft.tempo_key = dtp.tempo_key
    WHERE ft.marca_key <> -1 AND ft.tipo_key <> -1 AND ft.combustivel_key <> -1
    GROUP BY 1, 2, 3, 4, 5, 6
    ORDER BY 1, 2, 3, 4, 5, 6
"""

df_m1 = pd.read_sql(query, engine)
df_m1.head()


,marca_key,tipo_key,combustivel_key,localizacao_key,ano,mes,valor


In [3]:
# Código base para Auto-ARIMA
import pmdarima as pm
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Escolher uma série específica para testes
serie_exemplo = df_m1[(df_m1['marca_key']==1) & (df_m1['tipo_key']==1) & (df_m1['combustivel_key']==1)].copy()
serie_exemplo = serie_exemplo.sort_values(['ano','mes']).reset_index(drop=True)

# Transformação Box-Cox e Análise de Estacionariedade
# (Inspirado no Projeto_acabado.R)

print(f"Observações: {len(serie_exemplo)}")


Observações: 0
